# λ 조건부 키오스크 파레토 연구 — 새 실행 노트북

고정·상황별 **추천 가중치 함수**를 같은 학습 예산으로 비교합니다. 만족도 구성 가중치와 λ는 평가 조건이며 정책이 변경하지 못합니다. 기존 LinUCB/톰프슨 실험을 재개하는 파일이 아닙니다.

CPU부터 사용하세요. Pro+도 실행시간·자원을 무제한 보장하지 않습니다.

## 1. 특정 GitHub 버전 불러오기

비공개 저장소라면 왼쪽 열쇠 모양 Secrets에 `KIOSK_GITHUB_TOKEN`을 등록하고 노트북 접근을 허용하세요. 코드 읽기에는 해당 저장소 Contents 읽기, 결과 업로드에는 쓰기 권한이 필요합니다. 토큰을 코드나 채팅에 넣지 마세요.

재개 시에는 metadata.json의 정확한 커밋을 REVISION에 입력하세요.

In [ ]:
import os, re, sys, subprocess, tempfile
from pathlib import Path
REPOSITORY = "phj9187/koisk-simulation" #@param {type:"string"}
REVISION = "main" #@param {type:"string"}
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY):
    raise ValueError("저장소는 owner/name 형식이어야 합니다.")
if not re.fullmatch(r"[A-Za-z0-9_./-]+", REVISION) or REVISION.startswith("-"):
    raise ValueError("브랜치 또는 커밋 해시를 입력하세요.")
if any(n.startswith(("kiosk_research", "kiosk_sim", "kiosk_cloud")) for n in sys.modules):
    raise RuntimeError("이전 코드가 로드되어 있습니다. 런타임을 다시 시작한 뒤 실행하세요.")
token = None
try:
    from google.colab import userdata
    token = userdata.get("KIOSK_GITHUB_TOKEN")
except Exception:
    pass
CODE_ROOT = Path(tempfile.mkdtemp(prefix="kiosk-research-", dir="/content"))
with tempfile.TemporaryDirectory() as auth_dir:
    helper = Path(auth_dir) / "askpass.py"
    helper.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ.get('KIOSK_EPHEMERAL_TOKEN',''))\n")
    helper.chmod(0o700)
    child_env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS=str(helper))
    if token:
        child_env["KIOSK_EPHEMERAL_TOKEN"] = token
    try:
        commands = [
            ["git", "init", str(CODE_ROOT)],
            ["git", "-C", str(CODE_ROOT), "remote", "add", "origin", "https://github.com/"+REPOSITORY+".git"],
            ["git", "-C", str(CODE_ROOT), "fetch", "--depth", "1", "origin", REVISION],
            ["git", "-C", str(CODE_ROOT), "checkout", "--detach", "FETCH_HEAD"],
        ]
        for command in commands:
            result = subprocess.run(command, env=child_env, capture_output=True, text=True)
            if result.returncode:
                raise RuntimeError("GitHub 읽기 실패. 저장소·버전·비공개 저장소 권한을 확인하세요. 토큰은 출력하지 않습니다.")
    finally:
        child_env.pop("KIOSK_EPHEMERAL_TOKEN", None)
        token = None
COMMIT = subprocess.check_output(["git","-C",str(CODE_ROOT),"rev-parse","HEAD"],text=True).strip()
subprocess.check_call([sys.executable,"-m","pip","install","-q","-r",str(CODE_ROOT/"requirements.txt")])
sys.path.insert(0,str(CODE_ROOT/"src"))
print("코드 경로:", CODE_ROOT)
print("실행 커밋:", COMMIT)


## 2. Google Drive 인증·연결

아래 인증은 사용자 계정에서 승인합니다. Drive에는 원자료·모델·체크포인트·상세 결과를 저장합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from kiosk_research.persistence import layout
from kiosk_cloud.provenance import runtime_info
DATA_ROOT = layout("/content/drive/MyDrive/AI-Kiosk-Data")
print("저장 루트:", DATA_ROOT)
print(runtime_info())


## 3. 실험 조건·실행·안전 중지·재개

각 반복은 새 모델로 시작해 학습 → 동결 검증 → 동결 평가를 수행합니다. 반복 수 0은 중지 요청/런타임 종료까지 계속합니다. 기본값은 소규모 가상 검증용입니다. 실측 연구는 KREI 모드·64개 매장·원자료 경로를 선택하세요.

버튼은 백그라운드 작업자를 사용하므로 실행 셀이 반환된 뒤에도 중지 요청이 가능합니다. 버튼이 응답하지 않으면 Drive의 `experiments/실험ID/STOP` 파일을 만드세요. `RUNNING.lock`은 이전 실행이 끝났음을 확인한 뒤에만 삭제하세요.

원자료는 검증된 SHA256과 일치해야 합니다. 기존 `Kiosk_LHS/source_2025.xlsx` 경로를 직접 지정할 수 있습니다. 다른 자료나 가상 자료로 자동 대체하지 않습니다.

In [ ]:
from kiosk_research.ui import controls
panel = controls(DATA_ROOT, REPOSITORY)


## 4. 중지 완료 후 결과 확인 (선택)

결과 보기 버튼도 같은 기능입니다. 완전한 최종 평가 반복만 파레토·통계 비교에 포함됩니다. 5회 미만 반복은 기술통계만 제공합니다.

In [ ]:
run = panel["experiment"]
if run is not None:
    if panel["future"] is not None and not panel["future"].done():
        print("먼저 안전 중지를 요청하고 완료를 기다려주세요.")
    else:
        from kiosk_research.analysis import export
        summary = export(run)
        print("실험 ID:", run.id)
        print("완료 날짜:", run.state["day"])
        print("요약:", summary)


## 5. GitHub 요약 기록

업로드 버튼은 `results/실험ID-day-날짜` 새 브랜치만 만들며 기본 브랜치를 직접 수정하지 않습니다. 토큰이 없으면 건너뛰고 Drive에 요약을 보존합니다. 대용량 로그·모델·체크포인트는 GitHub에 올리지 않습니다.

여러 결과 브랜치를 기본 브랜치에 병합할 때 registry CSV 충돌을 정리해야 할 수 있습니다. 자세한 가정과 한계는 `docs/research_methodology_ko.md`, 실행 방법은 `docs/colab_research_ko.md`를 참고하세요.